In [3]:
import os
import datetime
import random
import numpy as np
import pandas as pd


from importlib import reload
from openai import OpenAI, RateLimitError
# from any_llm import completion, responses, list_models
from utils import constants, metrics, io_results, visualization
from utils.client import VotingProcessor
from utils.voting import VotingProbabilities, VotingPick
from utils.models_utils import check_response_api_support


## Experimen Setup

In [4]:
# Constants

# PROVIDER = "ollama"
# MODEL = "jean-luc/tiger-gemma-9b-v3:fp16"

PROVIDER = "openai"
MODEL = constants.GPT_41_NANO
# MODEL = constants.GPT_4o_MINI
# MODEL = constants.GPT_41
# MODEL = constants.GPT_4o

TEMPERATURE = 0

VOTING_RESULT = VotingProbabilities
# VOTING_RESULT = VotingPick

PROCESSOR = VotingProcessor()
# responses_api = check_response_api_support(PROVIDER)

In [7]:
# Loading Data

data = pd.read_csv("data/clean_data.csv", index_col=0)
from SoD.data_processing_utils import create_respondent_description
description_function = create_respondent_description

In [8]:
# Prompts

# instructions = """
# Jsi expertní AI asistent specializovaný na analýzu českého politického chování. Tvým úkolem je na základě demografického a postojového profilu respondenta odhadnout jeho volební chování ve volbách do Poslanecké sněmovny Parlamentu ČR v roce 2021.
#
# Tvůj výstup MUSÍ být JSON objekt, který přesně odpovídá Pydantic modelu `VotingResult`. Jiný formát není přípustný.
#
# Dodržuj tato pravidla:
# 1.  Analyzuj VŠECHNY poskytnuté informace o respondentovi (věk, vzdělání, bydliště, příjem, postoje k EU/NATO atd.).
# 2.  Na základě analýzy odhadni dvě klíčové věci:
#     a) Jaká je pravděpodobnost, že respondent vůbec šel k volbám.
#     b) Pokud volil, jaké jsou pravděpodobnosti pro jednotlivé politické strany.
# 3.  Vygeneruj JSON, který bude validní oproti poskytnutým Pydantic modelům.
# 4.  Dbej na to, aby součet pravděpodobností v objektu `voted_or_not` byl přesně 1.0.
# 5.  Dbej na to, aby součet pravděpodobností VŠECH stran v seznamu `parties` byl přesně 1.0."""

prompt_question = " Ve volbách do poslanecké sněmovny v roce 2021 jsem volil:"

## Prompt, Model and Response Showcase

In [9]:
respondent = data.iloc[143]
prompt = description_function(respondent) + prompt_question
prompt

'Jsem muž, je mi 43 let, mé vzdělání je základní + středoškolské vzdělání bez maturity. Žiji v Středočeském kraji, v okresu Beroun a obci o velikosti Méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 40.001 - 60.000 Kč. Nemám ani dobrou, ani špatnou životní úroveň. Spíše se nezajímám o politiku. Ve volbách do poslanecké sněmovny v roce 2021 jsem volil:'

In [10]:
# Pure GPT call
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
response = client.responses.parse(
    model=MODEL,
    temperature=TEMPERATURE,
    #instructions=INSTRUCTIONS,
    input=prompt,
    text_format=VOTING_RESULT
)

voted = response.output_parsed
response.output_parsed

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
# # any llm respons - Structure does not work
# response = responses(
#     provider=PROVIDER,
#     model=MODEL,
#     temperature=TEMPERATURE,
#     #instructions=INSTRUCTIONS,
#     input_data=prompt,
#     text=VOTING_RESULT
# )

In [ ]:
# # any llm completion - Only JSON
# response = completion(
#     provider=PROVIDER,
#     model=MODEL,
#     temperature=TEMPERATURE,
#     messages=[
#         # {"role": "system", "content": INSTRUCTIONS},
#         {"role": "user", "content": prompt }
#     ],
#     response_format=VOTING_RESULT
# )
# voted = VotingProbabilities.model_validate_json(response.choices[0].message.content)
# response.choices[0].message

In [7]:
p = 0
print(voted.voted_or_not)
for party_vote in voted.parties:
    print(f"{party_vote.name}: {party_vote.probability}")
    p += party_vote.probability
print(f"p_sum = {p}")

voted=0.0 not_voted=1.0


AttributeError: 'VotingPick' object has no attribute 'parties'

## Election Simulation Experiment

In [4]:
respondents = data
# respondents = data.sample(n=200, random_state=42)

In [ ]:
voting_results, skipped = PROCESSOR.run(
    data=respondents,
    prompt_creator= lambda res: description_function(res) + prompt_question,
    response_model=VOTING_RESULT,
    model= MODEL, temperature= TEMPERATURE
)

print(f"\nProcessed: {len(voting_results)}")
print(f"Failed/Skipped: {len(skipped)}")

In [ ]:
## Rerun skipped results
while len(skipped)>0:
    skipped_voting_results, skipped = PROCESSOR.run(
        data=respondents.iloc[skipped],
        prompt_creator= lambda res: description_function(res) + prompt_question,
        response_model=VOTING_RESULT,
        model= MODEL, temperature= TEMPERATURE
    )
    voting_results.update(skipped_voting_results)

## Save Results

In [ ]:
io_results.save_results_to_json(voting_results, f"results/voting_results_n={len(voting_results)}_t={TEMPERATURE}_{MODEL}_{VOTING_RESULT.__name__}.json")
io_results.save_results_to_json(voting_results, f"results/voting_results_n={len(voting_results)}_t={TEMPERATURE}_{MODEL}_{VOTING_RESULT.__name__}_{datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}.json")

In [7]:
voting_results_df = io_results.results_to_dataframe(voting_results)

In [ ]:
voting_results_df.to_csv(f"results/voting_results_n={len(voting_results_df)}_t={TEMPERATURE}_{MODEL}_{VOTING_RESULT.__name__}.csv", index=True)
voting_results_df.to_csv(f"results/voting_results_n={len(voting_results_df)}_t={TEMPERATURE}_{MODEL}_{VOTING_RESULT.__name__}_{datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}.csv", index=True)

## Result Evaluation

Validate the quality of the voting results by checking for common issues.

In [ ]:
# Evaluate voting results quality
tol = 0.01
bad_voted_sum, bad_party_probs_sum, duplicate_parties = metrics.evaluate_voting_results(voting_results, tol)

# Print details
if bad_voted_sum:
    print(f"[WARN] {len(bad_voted_sum)} respondents where voted+not_voted != 1 (|delta|>{tol}):")
    for rid, v, nv, s in bad_voted_sum[:20]:
        print(f" - ID={rid}: voted={v:.3f}, not_voted={nv:.3f}, sum={s:.3f}")
    if len(bad_voted_sum) > 20:
        print(f" ... and {len(bad_voted_sum) - 20} more")
else:
    print("[OK] All respondents have voted+not_voted summing to 1 within tolerance.")

if bad_party_probs_sum:
    print(f"[WARN] {len(bad_party_probs_sum)} respondents where party probabilities don't sum to 1 (|delta|>{tol}):")
    for rid, s, v in bad_party_probs_sum[:20]:
        print(f" - ID={rid}: voted={v}, prob_sum={s:.3f}")
    if len(bad_party_probs_sum) > 20:
        print(f" ... and {len(bad_party_probs_sum) - 20} more")
else:
    print("[OK] All respondents have party probabilities summing to 1 within tolerance (when parties present).")

if duplicate_parties:
    print(f"[WARN] {len(duplicate_parties)} respondents with duplicate party names:")
    for rid, dups in duplicate_parties[:20]:
        print(f" - ID={rid}: duplicates={dups}")
    if len(duplicate_parties) > 20:
        print(f" ... and {len(duplicate_parties) - 20} more")
else:
    print("[OK] No duplicate party names found in any respondent's parties list.")
# Visualize party probability errors
if bad_party_probs_sum:
    visualization.visualize_party_errors([abs(1 - s) for _, s, _ in bad_party_probs_sum])
else:
    print("No party probability errors to visualize.")

In [ ]:
# Visualize party probability errors
if bad_party_probs_sum:
    visualization.visualize_party_errors([abs(1-s) for _, s, _ in bad_party_probs_sum])
else:
    print("No party probability errors to visualize.")